# LLM as Judge: Evaluating BDD Test Case Quality

This notebook uses an LLM to evaluate the three types of BDD tests generated in `test_case_generation.ipynb`:
1. **Simple** – generated from app name only  
2. **With context** – generated from app name + app context  
3. **From graph** – generated from app screen graph (nodes/edges)

We define a rubric and evaluation criteria, then call the LLM to score and compare the test sets.

In [ ]:
# Configuration: app context and API key
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
# OpenAI API key (from .env OPENAI_API_KEY or environment variable)
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "your-api-key-here")

# App context: same as in test_case_generation.ipynb — used by the judge to assess relevance
APP_CONTEXT = """
A mobile banking app that allows users to view balance, transfer funds,
pay bills, and manage cards. Target: Android and iOS.
"""

# Paths to the three generated test case files (from test_case_generation.ipynb)
APP_NAME = "MyApp"
OUTPUT_DIR_SIMPLE = Path("test_cases_simple")
OUTPUT_DIR_CONTEXT = Path("test_cases_with_context")
OUTPUT_DIR_GRAPH = Path("test_cases_from_graph")

# Model used for LLM judge evaluation (e.g. gpt-4o-mini, gpt-4o)
MODEL_NAME = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")

## Rubric and evaluation criteria

The LLM judge scores each test set (1–5 per criterion) on:

| Criterion | Description | 1 (Poor) → 5 (Excellent) |
|-----------|-------------|---------------------------|
| **BDD structure** | Valid Given/When/Then flow; clear, testable steps; proper Gherkin style | Steps missing or vague; mixed or invalid structure |
| **Domain relevance** | Tests align with the app domain (e.g. banking: login, balance, transfer) | Generic or off-domain scenarios |
| **Coverage** | Breadth: launch, navigation, main flows, edge cases | Narrow or repetitive coverage |
| **Clarity & actionability** | Steps are unambiguous and executable by a human or automation | Vague or non-verifiable steps |
| **Completeness** | Each scenario has meaningful Given, When, Then; no trivial or empty steps | Incomplete or redundant scenarios |
| **Traceability** | Simple: fits app name; Context: reflects app context; Graph: maps to screen graph | No clear link to the source (name/context/graph) |

**Overall score** = average of the six criteria (1–5).  
The judge also returns a short justification and recommendation (which test type to prefer and why).

In [ ]:
# Load the three BDD test sets generated by test_case_generation.ipynb
import json

def load_test_cases(dir_path: Path, filename: str) -> list[dict]:
    path = dir_path / filename
    if not path.exists():
        return []
    with open(path) as f:
        return json.load(f)

name_safe = APP_NAME.replace(" ", "_")
simple = load_test_cases(OUTPUT_DIR_SIMPLE, f"{name_safe}_test_cases.json")
with_context = load_test_cases(OUTPUT_DIR_CONTEXT, f"{name_safe}_test_cases.json")
from_graph = load_test_cases(OUTPUT_DIR_GRAPH, f"{name_safe}_graph_test_cases.json")

print(f"Simple (app name only):     {len(simple)} feature(s)")
print(f"With context:               {len(with_context)} feature(s)")
print(f"From graph:                 {len(from_graph)} feature(s)")

In [ ]:
# LLM judge: evaluation prompt and criteria
from openai import OpenAI

if not OPENAI_API_KEY or OPENAI_API_KEY == "your-api-key-here":
    raise ValueError("Set OPENAI_API_KEY in the .env file or in your environment.")

client = OpenAI(api_key=OPENAI_API_KEY)

EVALUATION_CRITERIA = """
You are an expert QA engineer judging BDD (Given/When/Then) test case quality.

Score each test set from 1 (poor) to 5 (excellent) on these six criteria:

1. **BDD structure**: Valid Given/When/Then flow; clear, testable steps; proper Gherkin style.
2. **Domain relevance**: Tests align with the app domain (e.g. for a banking app: login, balance, transfer, cards).
3. **Coverage**: Breadth—launch, navigation, main flows, edge cases.
4. **Clarity & actionability**: Steps are unambiguous and executable by a human or automation.
5. **Completeness**: Each scenario has meaningful Given, When, Then; no trivial or empty steps.
6. **Traceability**: Tests clearly link to their source (app name only / app context / screen graph).

Respond with valid JSON only, in this exact shape (no markdown, no extra text):
{
  "evaluations": [
    {
      "test_type": "simple",
      "scores": { "bdd_structure": N, "domain_relevance": N, "coverage": N, "clarity_actionability": N, "completeness": N, "traceability": N },
      "overall": N,
      "summary": "One or two sentences."
    },
    { "test_type": "with_context", ... },
    { "test_type": "from_graph", ... }
  ],
  "recommendation": "Which test type to prefer and why (2-3 sentences)."
}
"""

In [ ]:
# Run the LLM judge
import re

def evaluate_bdd_tests(
    app_context: str,
    simple_tests: list,
    context_tests: list,
    graph_tests: list,
    model: str = None,
) -> dict:
    """Call LLM to evaluate the three BDD test sets; returns parsed evaluation JSON."""
    if model is None:
        model = MODEL_NAME
    user_content = f"""App context (what the app is):
{app_context.strip()}

---

Test set 1 — SIMPLE (generated from app name only):
{json.dumps(simple_tests, indent=2)}

---

Test set 2 — WITH CONTEXT (generated from app name + app context):
{json.dumps(context_tests, indent=2)}

---

Test set 3 — FROM GRAPH (generated from app screen graph):
{json.dumps(graph_tests, indent=2)}

---

Evaluate each of the three test sets using the six criteria. Return only the JSON object (no markdown)."""

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": EVALUATION_CRITERIA},
            {"role": "user", "content": user_content},
        ],
        temperature=0.2,
    )
    text = response.choices[0].message.content.strip()
    if "```json" in text:
        text = re.search(r"```(?:json)?\s*([\s\S]*?)```", text).group(1).strip()
    elif "```" in text:
        text = re.sub(r"^```\w*\s*", "", text).strip()
        text = re.sub(r"\s*```$", "", text).strip()
    return json.loads(text)

In [ ]:
# Evaluate the three BDD test sets
result = evaluate_bdd_tests(
    app_context=APP_CONTEXT,
    simple_tests=simple,
    context_tests=with_context,
    graph_tests=from_graph,
    model=MODEL_NAME,
)

# Pretty-print evaluations
for ev in result["evaluations"]:
    print(f"\n=== {ev['test_type'].upper().replace('_', ' ')} ===")
    print(f"Overall: {ev['overall']}/5")
    print(f"Scores: {ev['scores']}")
    print(f"Summary: {ev['summary']}")

print(f"\n--- Recommendation ---\n{result['recommendation']}")

In [ ]:
# Full evaluation result (for inspection or downstream use)
result